# Cross-Domain QLoRA — v3 (No SFTTrainer)
**Change DOMAIN in cell 2.** Uses plain PyTorch loop — no trl dependency issues.

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate scikit-learn

In [ ]:
DOMAIN = "ag_news"  # "ag_news" or "go_emotions" or "ledgar"
SEED = 42
TRAIN_SIZE = 5000
TEST_SIZE = 1000
MAX_LEN = 384
EPOCHS = 3
LR = 2e-4
BATCH = 4
GRAD_ACC = 8
LORA_R = 64
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

import torch, random, math, json, time
from collections import Counter
random.seed(SEED)
torch.manual_seed(SEED)
print(f"GPU: {torch.cuda.get_device_name(0)} | Domain: {DOMAIN}")

In [ ]:
from datasets import load_dataset

if DOMAIN == "ag_news":
    ds = load_dataset("ag_news")
    LABELS = ["World","Sports","Business","Sci/Tech"]
    SYS = "Classify this news. Reply with ONLY: World, Sports, Business, or Sci/Tech"
    def get(ex): return ex["text"], LABELS[ex["label"]]
    tr_s, te_s = "train", "test"
elif DOMAIN == "go_emotions":
    ds = load_dataset("google-research-datasets/go_emotions","simplified")
    LABELS = ["admiration","amusement","anger","annoyance","approval","caring","confusion","curiosity","desire","disappointment","disapproval","disgust","embarrassment","excitement","fear","gratitude","grief","joy","love","nervousness","neutral","optimism","pride","realization","relief","remorse","sadness","surprise"]
    SYS = "Identify the emotion. Reply with ONLY the emotion name."
    def get(ex): return ex["text"], LABELS[ex["labels"][0]]
    tr_s, te_s = "train", "test"
elif DOMAIN == "ledgar":
    ds = load_dataset("lex_glue","ledgar")
    LABELS = ds["train"].features["label"].names
    SYS = "Classify this legal provision. Reply with ONLY the provision type."
    def get(ex): return ex["text"], LABELS[ex["label"]]
    tr_s, te_s = "train", "validation"

raw_tr = ds[tr_s].shuffle(seed=SEED).select(range(min(TRAIN_SIZE,len(ds[tr_s]))))
raw_te = ds[te_s].shuffle(seed=SEED).select(range(min(TEST_SIZE,len(ds[te_s]))))
lc = Counter([get(ex)[1] for ex in raw_tr]); tot=sum(lc.values())
H = -sum((c/tot)*math.log2(c/tot) for c in lc.values())
print(f"{DOMAIN}: {len(LABELS)} classes, H={H:.2f}, train={len(raw_tr)}, test={len(raw_te)}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tok.pad_token = tok.eos_token
tok.padding_side = "right"

mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True),
    device_map="auto", trust_remote_code=True)
mdl = prepare_model_for_kbit_training(mdl)
mdl = get_peft_model(mdl, LoraConfig(r=LORA_R, lora_alpha=LORA_R*2,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0, bias="none", task_type="CAUSAL_LM"))
mdl.print_trainable_parameters()

In [ ]:
# Tokenize training data
def tokenize(ex):
    txt, lbl = get(ex)
    msgs = [{"role":"system","content":SYS},{"role":"user","content":txt[:300]},{"role":"assistant","content":lbl}]
    s = tok.apply_chat_template(msgs, tokenize=False)
    enc = tok(s, truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
    enc["labels"] = enc["input_ids"].clone()
    enc["labels"][enc["attention_mask"]==0] = -100
    return {k:v.squeeze(0) for k,v in enc.items()}

train_tok = raw_tr.map(tokenize, remove_columns=raw_tr.column_names)
train_tok.set_format("torch")
print(f"Tokenized {len(train_tok)} samples")

In [ ]:
# Train with PyTorch — no SFTTrainer needed!
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

loader = DataLoader(train_tok, batch_size=BATCH, shuffle=True)
opt = torch.optim.AdamW(mdl.parameters(), lr=LR, weight_decay=0.01)
total_steps = (len(loader) // GRAD_ACC) * EPOCHS
sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=10, num_training_steps=total_steps)

mdl.train()
t0 = time.time()
for epoch in range(EPOCHS):
    total_loss = 0
    opt.zero_grad()
    for step, batch in enumerate(loader):
        batch = {k:v.to("cuda") for k,v in batch.items()}
        loss = mdl(**batch).loss / GRAD_ACC
        loss.backward()
        total_loss += loss.item() * GRAD_ACC
        if (step+1) % GRAD_ACC == 0:
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        if (step+1) % 100 == 0:
            print(f"  E{epoch+1} step {step+1}/{len(loader)} loss={total_loss/(step+1):.4f}")
    print(f"Epoch {epoch+1}/{EPOCHS} loss={total_loss/len(loader):.4f}")

train_min = (time.time()-t0)/60
print(f"\nDone in {train_min:.1f} min")

In [ ]:
# Evaluate
from sklearn.metrics import f1_score
from tqdm import tqdm

mdl.eval()
preds, golds, raws = [], [], []
for ex in tqdm(raw_te, desc="Eval"):
    txt, lbl = get(ex)
    msgs = [{"role":"system","content":SYS},{"role":"user","content":txt[:300]}]
    p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(p, return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=32, do_sample=False)
    resp = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    preds.append(resp); golds.append(lbl)
    raws.append({"label":lbl,"predict":resp})

f1 = f1_score(golds, preds, average="macro", zero_division=0)
acc = sum(p==l for p,l in zip(preds,golds))/len(golds)
print(f"\n{'='*50}")
print(f"{DOMAIN} | H={H:.2f} | F1={f1:.4f} | Acc={acc:.4f} | {train_min:.1f}min")
print(f"{'='*50}")

res = {"domain":DOMAIN,"entropy":round(H,2),"f1":round(f1,4),"acc":round(acc,4),
       "train_min":round(train_min,1),"classes":len(LABELS),"model":MODEL_ID,"rank":LORA_R}
with open(f"/kaggle/working/{DOMAIN}_results.json","w") as f: json.dump(res,f,indent=2)
print(f"Saved: /kaggle/working/{DOMAIN}_results.json")

In [ ]:
for r in raws[:15]:
    s = "Y" if r["label"]==r["predict"] else "N"
    print(f"{s} | {r['label']:20s} | {r['predict'][:30]}")